# Research assistant trace

Runs the six-agent workflow on two provided papers. The model is scripted, so this notebook does not call Gemini. Each line is one agent calling one tool on the shared dossier. The same pipeline with the live model is `python -m research_assistant`.


In [1]:
from research_assistant.demo import run_offline_demo

dossier = await run_offline_demo()
print("Multi-agent trace")
for line in dossier.trace:
    print(line)
print()
print(dossier.report_markdown)


Multi-agent trace
query · list_corpus_titles · ok · - Paper A - Paper B
query · commit_subqueries · ok · alpha measured error | alpha does not reduce | passage retrieved
retrieval · embed_subqueries · ok · lexical fallback
retrieval · store_retrieved_passages · ok · 2 passages: paper-a#c0, paper-b#c0
summary · read_passages_for_summary · ok · [paper-a#c0] Paper A Alpha reduces measured error when the passage is retrieved and shown. The lab recorded the drop. [paper-b#c0] Paper B Alpha does not reduce measured error. Th…
contradiction · read_passages_for_contradictions · ok · [paper-a#c0] Paper A Alpha reduces measured error when the passage is retrieved and shown. The lab recorded the drop. [paper-b#c0] Paper B Alpha does not reduce measured error. Th…
summary · commit_claims · ok · 2 claims
contradiction · commit_contradictions · ok · 1 contradictions
factcheck · select_checkable_claims · ok · {"eligible": [{"claim_id": "c1", "theme": "Measured effect", "text": "Alpha reduces measured

## A question the papers do not cover

The run stops before any agent. The refusal is appended to `logs/run.log`, and the lines above stay in the file.


In [2]:
from research_assistant.agents.retrieval_agent import RetrievalError
from research_assistant.demo import LOG_PATH, run_offline_demo

before = LOG_PATH.read_text(encoding="utf-8").splitlines()
try:
    await run_offline_demo("zebrafish sonnet form")
except RetrievalError as exc:
    print(f"error: {exc}")
after = LOG_PATH.read_text(encoding="utf-8").splitlines()
print(f"Log grew from {len(before)} to {len(after)} lines.")
print("First line unchanged:", after[0] == before[0])
print(after[-1])


error: These papers do not cover that question. The assistant will not invent sources.
Log grew from 33 to 34 lines.
First line unchanged: True
{"ts": "2026-11-06T14:05:03.089795+00:00", "run_id": "559b61fc", "agent": "pipeline", "tool": "corpus_covers", "ok": false, "detail": "These papers do not cover that question. The assistant will not invent sources.", "dossier_id": 4523255248}
